In [0]:
%run Databricks/WB-Project/jinja_gold_query_creation.ipynb

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS walmart.gold

In [0]:

for i in parameters:

    if i['name'].startswith('dim'):
        creation_query_text = Template(
            jinja_creation_query_dim
        ).render(param=i)

        spark.sql(creation_query_text)

        creation_new_query_text = Template(
            jinja_new_versions_query_dim
        ).render(param=i)

        spark.sql(creation_new_query_text)

        creation_expire_query_text = Template(
            jinja_expire_query_dim
        ).render(param=i)

        spark.sql(creation_expire_query_text)

        insertion_query_text = Template(
            jinja_insert_query_dim
        ).render(param=i)
        spark.sql(insertion_query_text)

    elif i['name'].startswith('fact'):

        fact_query_text = Template(
            jinja_fact_query
        ).render(param=i)

        spark.sql(fact_query_text).write.format('delta') \
            .mode('overwrite') \
            .option('overwriteSchema', 'true') \
            .option('path', f"{gold_location}/{i['name']}") \
            .saveAsTable(f"walmart.gold.{i['name']}")